## 학습 안내

이 노트북은 **단어 벡터와 의미 공간**를 파이썬 개발자가 torch 코드 흐름으로 읽을 수 있게 구성한 실습입니다.
코드를 볼 때는 먼저 데이터의 shape을 확인하고, 그 다음 `nn.Module` 계층이 어떤 텐서를 받아 어떤 텐서를 반환하는지 따라가면 좋습니다.

**학습 목표**
- 단어가 숫자 벡터로 표현되는 이유를 이해합니다.
- 유사도와 벡터 산술이 의미 관계를 어떻게 드러내는지 확인합니다.
- 이후 토큰 임베딩과 Transformer 입력의 직관을 준비합니다.

**torch 읽기 포인트**
- 임베딩은 torch Tensor와 같은 밀집 숫자 벡터로 생각할 수 있습니다.
- 유사도 계산은 벡터 내적과 정규화의 조합입니다.


## Gensim 실습 예제
아래 코드는 `gensim`으로 단어 벡터를 만들고, 의미 계산/유사 단어 탐색/단어 그룹 분류를 수행합니다.


In [1]:
# 학습 가이드
# - 셀 목적: vector 노트북의 핵심 단계 3을 실행합니다.
# - torch 관점: 임베딩은 torch Tensor와 같은 밀집 숫자 벡터로 생각할 수 있습니다.
# - 확인 포인트: 실행 후 출력 텐서의 shape과 값의 의미를 함께 확인하세요.

%pip install gensim


  Using cached gensim-4.4.0-cp312-cp312-win_amd64.whl.metadata (8.6 kB)
  Using cached smart_open-7.5.1-py3-none-any.whl.metadata (24 kB)
Using cached gensim-4.4.0-cp312-cp312-win_amd64.whl (24.4 MB)
Using cached smart_open-7.5.1-py3-none-any.whl (64 kB)
Note: you may need to restart the kernel to use updated packages.


In [9]:
# 학습 가이드
# - 셀 목적: 실습에 필요한 외부 데이터나 가중치를 내려받고 로컬 파일로 준비합니다.
# - torch 관점: 임베딩은 torch Tensor와 같은 밀집 숫자 벡터로 생각할 수 있습니다.
# - 확인 포인트: 실행 후 출력 텐서의 shape과 값의 의미를 함께 확인하세요.

import gensim.downloader as api

def run_gensim_tutorial():
    # 1. 사전 학습된 가벼운 모델 다운로드 및 로드 (약 65MB)
    # 강의 중 첫 실행 시 다운로드 시간이 약간 소요될 수 있습니다.
    print("모델을 로딩 중입니다... (glove-wiki-gigaword-50)\n")
    model = api.load("glove-wiki-gigaword-50")
    
    # ----------------------------------------------------
    # [기능 1] 단어의 의미(유사도) 계산하기
    # 두 단어 벡터 간의 코사인 유사도(Cosine Similarity)를 계산합니다.
    # ----------------------------------------------------
    print("1. 단어 간 유사도 계산하기")
    word1, word2, word3 = 'cat', 'dog', 'car'
    
    sim_cat_dog = model.similarity(word1, word2)
    sim_cat_car = model.similarity(word1, word3)
    
    print(f" - '{word1}'와 '{word2}'의 유사도: {sim_cat_dog:.4f}")
    print(f" - '{word1}'와 '{word3}'의 유사도: {sim_cat_car:.4f}\n")
    
    # ----------------------------------------------------
    # [기능 2] 유사한 단어 가져오기
    # 특정 단어와 벡터 공간상에서 가장 가까운 단어들을 추출합니다.
    # ----------------------------------------------------
    print("2. 'computer'와 가장 유사한 단어 5개 가져오기")
    target_word = 'computer'
    similar_words = model.most_similar(target_word, topn=5)
    
    for word, score in similar_words:
        print(f" - {word} (유사도 점수: {score:.4f})")
    print()
    
    # ----------------------------------------------------
    # [기능 3] 주어진 단어들을 그룹으로 분류 (이질적인 단어 찾기)
    # 단어들의 의미적 군집을 파악하여, 그룹에 어울리지 않는 단어를 찾아냅니다.
    # ----------------------------------------------------
    print("3. 단어 그룹 중 성격이 다른 단어(Outlier) 분류하기")
    word_group = ['apple', 'banana', 'orange', 'car']
    
    outlier = model.doesnt_match(word_group)
    print(f" - 단어 그룹: {word_group}")
    print(f" - 과일 그룹에 어울리지 않는 단어: '{outlier}'\n")

# 실행
if __name__ == "__main__":
    run_gensim_tutorial()

모델을 로딩 중입니다... (glove-wiki-gigaword-50)

1. 단어 간 유사도 계산하기
 - 'cat'와 'dog'의 유사도: 0.9218
 - 'cat'와 'car'의 유사도: 0.3638

2. 'computer'와 가장 유사한 단어 5개 가져오기
 - computers (유사도 점수: 0.9165)
 - software (유사도 점수: 0.8815)
 - technology (유사도 점수: 0.8526)
 - electronic (유사도 점수: 0.8126)
 - internet (유사도 점수: 0.8060)

3. 단어 그룹 중 성격이 다른 단어(Outlier) 분류하기
 - 단어 그룹: ['apple', 'banana', 'orange', 'car']
 - 과일 그룹에 어울리지 않는 단어: 'car'



## Online 실습 예제
백터공간에서 단어 의미를 시뮬레이션합니다.

- https://www.cs.cmu.edu/~dst/WordEmbeddingDemo/
- Word Embedding Demo
- Soldier는 어느 위치일까?

- https://vectors.nlpl.eu/explore/embeddings/en/calculator/
- Korea_PROPN, Busan_PROPN 을 입력한 후 Japan_PROPN을 입력하면 뭐가 나올까요?

- https://vectors.nlpl.eu/explore/embeddings/en/associates/#
- mother을 입력하면 연관된 단어는 어떤 것이 나올까요?

- https://vectors.nlpl.eu/explore/embeddings/en/visual/
- car,tank,transport,computer,mouse,Moscow,Paris,London,Russia,France,Britain,clean,dirty,new 를 입력하면 어떻게 분류가 될까요?